In [ ]:
import os
import numpy as np
import json
import re
from dataclasses import dataclass, field
from typing import Optional, Union
from collections import defaultdict
from matplotlib import pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter
from tabulate import tabulate


@dataclass
class LineData:
    model: str
    dataset: str
    slo_settings: Union[tuple[float, float], dict[str, tuple[float, float]]]
    method_name: str
    request_rates: list[float]
    n_output_tokens: list[int]
    avg_latency: list[float]
    p90_latency: list[float]
    p99_latency: list[float]
    avg_ttft: list[float]
    p90_ttft: list[float]
    p99_ttft: list[float]
    avg_tpot: list[float]
    p90_tpot: list[float]
    p99_tpot: list[float]
    ttft_slo_attainment: list[float]
    tpot_slo_attainment: list[float]
    slo_attainment: list[float]
    
    dataset_to_avg_latency: dict[str, list[float]]
    dataset_to_p90_latency: dict[str, list[float]]
    dataset_to_p99_latency: dict[str, list[float]]
    dataset_to_avg_ttft: dict[str, list[float]]
    dataset_to_p90_ttft: dict[str, list[float]]
    dataset_to_p99_ttft: dict[str, list[float]]
    dataset_to_avg_tpot: dict[str, list[float]]
    dataset_to_p90_tpot: dict[str, list[float]]
    dataset_to_p99_tpot: dict[str, list[float]]
    dataset_to_ttft_slo_attainment: dict[str, list[float]]
    dataset_to_tpot_slo_attainment: dict[str, list[float]]
    dataset_to_slo_attainment: dict[str, list[float]]
    
    goodput: float = 0

@dataclass
class ResultAnalyzerConfig:
    base_dir: str = os.getcwd()
    result_dir_pattern: str = r'^\d{8}_\d{6}$' # Regex pattern to match folders like "20250425_080607"
     # we only analyze data of model in model_filter, dataset in dataset filter, analyze all if set None
    model_filter: Optional[list[str]] = None
    dataset_filter: Optional[list[str]] = None
    method_name_filter: Optional[list[str]] = None
    # used to plot label in figure
    model_label_map: dict[str, str] = field(default_factory=dict)
    mixed_dataset_label: str = 'Mixed'
    dataset_label_map: dict[str, str] = field(default_factory=dict)
    method_name_label_map: dict[str, str] = field(default_factory=dict)
    candidate_ttft_slos: list[float] = field(default_factory=dict)
    candidate_tpot_slos: list[float] = field(default_factory=dict)
    goodput_slo_target: float = 0.9
    force_slo_settings: dict[tuple[str, str], tuple[float, float]] = field(default_factory=dict) # (model, dataset) -> (ttft, tpot)
    
    
class ResultAnalyzer:
    """
    ResultAnalyzer is a builder class of paper figure and paper table in latex format
    """
    def __init__(self, config: ResultAnalyzerConfig):
        self.config = config
        self._create_dir()
        self.result_dir_pattern = re.compile(self.config.result_dir_pattern)
        
        self.all_json_data: list[dict] = []
        self.models = set()
        self.datasets = set()
        self.method_names = set()
        self.model_dataset_to_json_data = defaultdict(list) # model: str, dataset: str -> list[log]
        self.all_line_data: list[LineData] = [] # all model, dataset, tpot slo, ttft slo, methods' line data
        self.model_dataset_ttft_tpot_method_to_line_data = defaultdict(list)
        self.model_dataset_ttft_tpot_method_to_best_line_data: dict[tuple[str, str, float, float, str], LineData] = {}
        self.all_plot_line_data: list[LineData] = []
        self.best_slo_settings: dict[tuple[str, str], tuple[float, float]] = {}
        
    def _create_dir(self):
        self.results_dir = os.path.join(self.config.base_dir, 'result')
        self.figures_dir = os.path.join(self.config.base_dir, 'figure')
        self.tables_dir = os.path.join(self.config.base_dir, 'table')
        os.makedirs(self.tables_dir, exist_ok=True)
        os.makedirs(self.figures_dir, exist_ok=True)
        
    def load_result_json_data(self, results_path: Optional[str]=None, recursive: bool = True):
        if results_path is None:
            results_path = self.results_dir 
        for folder_name in os.listdir(results_path):
            folder_path = os.path.join(results_path, folder_name)
            if not os.path.isdir(folder_path):
                continue
            if self.result_dir_pattern.match(folder_name):
                for file_name in os.listdir(folder_path):
                    if file_name.endswith('.json'):
                        file_path = os.path.join(folder_path, file_name)
                        try:
                            with open(file_path, 'r', encoding='utf-8') as f:
                                data = json.load(f)
                                print(f"loaded result json data from {file_path}")
                                self.all_json_data.append(data)
                        except Exception as e:
                            print(f"Error reading file {file_path}: {e}")
            elif recursive:
                self.load_result_json_data(folder_path, recursive=recursive)
        
    def log(self):
        print(f"number of json file {len(self.all_json_data)}")
        n_models = len(self.models)
        n_datasets = len(self.datasets)
        n_method_name = len(self.method_names)
        print(f'number of models {n_models}')
        print(f'models: {self.models}')
        print(f'number of datasets {n_datasets}')
        print(f'datasets {self.datasets}')
        print(f'method_names {self.method_names}')
        print(f'number of methods {n_method_name}')
        for (model, dataset), methods_data in self.model_dataset_to_json_data.items():
            model_dataset_to_method_names = [method_data['method_name'] for method_data in methods_data]
            print(f'{model}, {dataset} -> {model_dataset_to_method_names}')
        
    def classify_filter_model(self):
        self.models = set()
        self.datasets = set()
        self.method_names = set()
        self.model_dataset_to_json_data = defaultdict(list) # model: str, dataset: str -> list[log]
        for json_data in self.all_json_data:
            # 1. load model and dataset and method
            model = json_data['model']
            dataset_weights: dict[str, int] = json_data['datasets']
            dataset: str = ""
            if sum(1 for v in json_data['datasets'].values() if v != 0) > 1:
                dataset = self.config.mixed_dataset_label
            else:
                dataset = next(k for k, v in json_data['datasets'].items() if v != 0)
            method_name = json_data['method_name']
                
            # 2. select certain models and datasets used to analyze data
            def filter(name: str, filter: Optional[list[str]]):
                return not self.config.model_filter or model in self.config.model_filter
                
            if not filter(model, self.config.model_filter) or not filter(dataset, self.config.dataset_filter) or not filter(method_name, self.config.method_name_filter):
                continue

            def map_name_to_label(name: str, label_map: dict, json_data: dict, json_key: str) -> str:
                if name in label_map:
                    name = label_map
                json_data[f'{json_key}_label'] = name
                return name
            map_name_to_label(model, self.config.model_label_map, json_data, 'model')
            map_name_to_label(dataset, self.config.dataset_label_map, json_data, 'dataset')
            map_name_to_label(method_name, self.config.method_name_label_map, json_data, 'method_name')
            
            # 4. statstic model and 
            self.models.add(model)
            self.datasets.add(dataset)
            self.method_names.add(method_name)
            self.model_dataset_to_json_data[model, dataset].append(json_data)

    def get_intersection_points(self, x: list[float], y: list[float], target_y: float) -> list[tuple[float, float]]:
        cross_points: list[tuple[float, float]] = []
        # Check if y_i is close to target_y
        for xi, yi in zip(x, y):
            if np.isclose(yi, target_y):
                cross_points.append((xi, yi))
        # Check if each line crosses the y=target_y line
        for i in range(len(x) - 1):
            x0, x1 = x[i], x[i + 1]
            y0, y1 = y[i], y[i + 1]
            if (y0 - target_y) * (y1 - target_y) < 0:
                t = (target_y - y0) / (y1 - y0)
                x_inter = x0 + t * (x1 - x0)
                y_inter = y0 + t * (y1 - y0)
                cross_points.append((x_inter, y_inter))
        return cross_points

    def compute_goodput(self, line_data: LineData):
        x = line_data.request_rates
        y = line_data.slo_attainment
        cross_points = self.get_intersection_points(x, y, self.config.goodput_slo_target)
        if not cross_points:
            if y[0] < self.config.goodput_slo_target:
                line_data.goodput = x[0]
            elif y[-1] > self.config.goodput_slo_target:
                line_data.goodput = x[-1]
        else:
            line_data.goodput = cross_points[0][0]

        def make_line_data(self, dataset_to_latency: dict[str, list[float]], dataset_to_ttft: dict[str, list[float]], dataset_to_tpot: dict[str, list[float]]) -> LineData:
            pass
            
    def compute_requests_metric(self, outputs: list[dict]) -> tuple[dict[str, list[float]], dict[str, list[float]], dict[str, list[float]]]:
        failed_request = 0
        dataset_to_ttfts: dict[str, list[float]] = defaultdict(list)
        dataset_to_tpots: dict[str, list[float]] = defaultdict(list)
        dataset_to_latencies: dict[str, list[float]] = defaultdict(list)
        for output in outputs:
            if not output['success']:
                failed_request += 1
                continue
            if len(output['token_times']) <= 1:
                failed_request += 1
                continue

            dataset = output['entry']['dataset']
            start_time = output['start_time']
            token_times = output['token_times']
            latency = token_times[-1] - start_time
            ttft = token_times[0] - start_time
            tpots = [token_times[i] - token_times[i - 1] for i in range(1, len(token_times))]
            dataset_to_ttfts[dataset].append(ttft)
            dataset_to_tpots[dataset].extend(tpots)
            dataset_to_latencies[dataset].append(latency)
            
        return dataset_to_ttfts, dataset_to_tpots, dataset_to_latencies

    def compute_line_data(self, json_data: dict, slo_settings: Union[tuple[float, float], dict[str, tuple[float, float]]]) -> LineData:
        request_rates: list[float] = []
        n_output_tokens: list[int] = []
        avg_latency: list[float] = []
        p90_latency: list[float] = []
        p99_latency: list[float] = []
        avg_ttft: list[float] = []
        p90_ttft: list[float] = []
        p99_ttft: list[float] = []
        avg_tpot: list[float] = []
        p90_tpot: list[float] = []
        p99_tpot: list[float] = []
        tpot_slo_attainment: list[float] = []
        ttft_slo_attainment: list[float] = []
        slo_attainment: list[float] = []
        
        dataset_to_avg_latency: dict[str, list[float]] = defaultdict(list)
        dataset_to_p90_latency: dict[str, list[float]] = defaultdict(list)
        dataset_to_p99_latency: dict[str, list[float]] = defaultdict(list)
        dataset_to_avg_ttft: dict[str, list[float]] = defaultdict(list)
        dataset_to_p90_ttft: dict[str, list[float]] = defaultdict(list)
        dataset_to_p99_ttft: dict[str, list[float]] = defaultdict(list)
        dataset_to_avg_tpot: dict[str, list[float]] = defaultdict(list)
        dataset_to_p90_tpot: dict[str, list[float]] = defaultdict(list)
        dataset_to_p99_tpot: dict[str, list[float]] = defaultdict(list)
        dataset_to_ttft_slo_attainment: dict[str, list[float]] = defaultdict(list)
        dataset_to_tpot_slo_attainment: dict[str, list[float]] = defaultdict(list)
        dataset_to_slo_attainment: dict[str, list[float]] = defaultdict(list)
        
        results = json_data['results']
        for result in results: # iterate over each request rate
            outputs = result['outputs']
            if "dataset_to_latencies" in result: # if we caculated all ttfts before
                dataset_to_latencies = result["dataset_to_latencies"]
                dataset_to_ttfts = result["dataset_to_ttfts"]
                dataset_to_tpots = result["dataset_to_tpots"]
            else:
                dataset_to_ttfts, dataset_to_tpots, dataset_to_latencies = self.compute_requests_metric(outputs)
                result["dataset_to_latencies"] = dataset_to_latencies
                result["dataset_to_ttfts"] = dataset_to_ttfts
                result["dataset_to_tpots"] = dataset_to_tpots

            all_ttfts = sum(dataset_to_ttfts.values(), [])
            all_tpots = sum(dataset_to_tpots.values(), [])
            all_latencies = sum(dataset_to_latencies.values(), [])
            sum_output_tokens = sum(all_tpots) + sum(all_ttfts)
            
            ttft_satisfied_cnt = 0
            tpot_satisfied_cnt = 0
            satisfied_cnt = 0
            if isinstance(slo_settings, tuple):
                TTFT_SLO, TPOT_SLO = slo_settings
                for ttft, tpot in zip(all_ttfts, all_tpots):
                    ttft_satisfied = ttft < TTFT_SLO
                    tpot_satisfied = tpot < TPOT_SLO
                    satisfied = ttft_satisfied and tpot_satisfied
                    ttft_satisfied_cnt += ttft_satisfied
                    tpot_satisfied_cnt += tpot_satisfied
                    satisfied_cnt += satisfied
            elif isinstance(slo_settings, dict):
                for dataset in dataset_to_latencies.keys():
                    TTFT_SLO, TPOT_SLO = slo_settings[dataset]
                    for ttft, tpot in zip(dataset_to_ttfts[dataset], dataset_to_tpots[dataset]):
                        ttft_satisfied = ttft < TTFT_SLO
                        tpot_satisfied = tpot < TPOT_SLO
                        satisfied = ttft_satisfied and tpot_satisfied
                        ttft_satisfied_cnt += ttft_satisfied
                        tpot_satisfied_cnt += tpot_satisfied
                        satisfied_cnt += satisfied
            else:
                raise Exception('invalid slo_settings')

            time_out_latency = 60
            total_requests = len(outputs)
            request_rate = result['request_rate']
            request_rates.append(request_rate)
            n_output_tokens.append(sum_output_tokens)

            
            if len(all_latencies) == 0:
                latency_metrics = [avg_latency, p90_latency, p99_latency, avg_ttft, p90_ttft, p99_ttft, avg_tpot, p90_tpot, p99_tpot]
                slo_metrices = [ttft_slo_attainment, tpot_slo_attainment, slo_attainment]
                dataset_to_metrics = [dataset_to_avg_latency, dataset_to_p90_latency, dataset_to_p99_latency, dataset_to_avg_ttft, dataset_to_p90_ttft, dataset_to_p99_ttft, dataset_to_avg_tpot, dataset_to_p90_tpot, dataset_to_p99_tpot]
                dataset_to_slo_metrices = [dataset_to_ttft_slo_attainment, dataset_to_tpot_slo_attainment, dataset_to_slo_attainment]
                for metric in latency_metrics:
                    metric.append(time_out_latency)
                for metric in slo_metrices:
                    metric.append(0)
                for dataset_to_metric in dataset_to_metrics:
                    for datset, metric in dataset_to_metric.items():
                        metric.append(time_out_latency)
                for dataset_to_metric in dataset_to_slo_metrices:
                    for datset, metric in dataset_to_metric.items():
                        metric.append(0)
            else:
                avg_latency.append(np.mean(all_latencies))
                p90_latency.append(np.percentile(all_latencies, 90))
                p99_latency.append(np.percentile(all_latencies, 99))
                avg_ttft.append(np.mean(all_ttfts))
                p90_ttft.append(np.percentile(all_ttfts, 90))
                p99_ttft.append(np.percentile(all_ttfts, 99))
                avg_tpot.append(np.mean(all_tpots))
                p90_tpot.append(np.percentile(all_tpots, 90))
                p99_tpot.append(np.percentile(all_tpots, 99))
                ttft_slo_attainment.append(ttft_satisfied_cnt / total_requests)
                tpot_slo_attainment.append(tpot_satisfied_cnt / total_requests)
                slo_attainment.append(satisfied_cnt / total_requests)
                for dataset, latencies in dataset_to_latencies.items():
                    dataset_to_avg_latency[dataset].append(np.mean(latencies))
                    dataset_to_p90_latency[dataset].append(np.percentile(latencies, 90))
                    dataset_to_p99_latency[dataset].append(np.percentile(latencies, 99))
                for dataset, ttfts in dataset_to_ttfts.items():
                    dataset_to_avg_ttft[dataset].append(np.mean(ttfts))
                    dataset_to_p90_ttft[dataset].append(np.percentile(ttfts, 90))
                    dataset_to_p99_ttft[dataset].append(np.percentile(ttfts, 99))
                for dataset, tpots in dataset_to_tpots.items():
                    dataset_to_avg_tpot[dataset].append(np.mean(tpots))
                    dataset_to_p90_tpot[dataset].append(np.percentile(tpots, 90))
                    dataset_to_p99_tpot[dataset].append(np.percentile(tpots, 99))
                    
        line_data = LineData(
            model               = json_data['model'],
            dataset             = json_data['dataset_label'],
            slo_settings        = slo_settings, 
            method_name         = json_data['method_name'],
            request_rates       = request_rates,
            n_output_tokens     = n_output_tokens,
            avg_latency         = avg_latency,
            p90_latency         = p90_latency,
            p99_latency         = p99_latency,
            avg_ttft            = avg_ttft,
            p90_ttft            = p90_ttft,
            p99_ttft            = p99_ttft,
            avg_tpot            = avg_tpot,
            p90_tpot            = p90_tpot,
            p99_tpot            = p99_tpot,
            ttft_slo_attainment = ttft_slo_attainment,
            tpot_slo_attainment = tpot_slo_attainment,
            slo_attainment      = slo_attainment,

            dataset_to_avg_latency = dataset_to_avg_latency, 
            dataset_to_p90_latency = dataset_to_p90_latency, 
            dataset_to_p99_latency = dataset_to_p99_latency, 
            dataset_to_avg_ttft = dataset_to_avg_ttft, 
            dataset_to_p90_ttft = dataset_to_p90_ttft, 
            dataset_to_p99_ttft = dataset_to_p99_ttft, 
            dataset_to_avg_tpot = dataset_to_avg_tpot, 
            dataset_to_p90_tpot = dataset_to_p90_tpot, 
            dataset_to_p99_tpot = dataset_to_p99_tpot, 
            dataset_to_ttft_slo_attainment = dataset_to_ttft_slo_attainment, 
            dataset_to_tpot_slo_attainment = dataset_to_tpot_slo_attainment, 
            dataset_to_slo_attainment = dataset_to_slo_attainment, 
        )
        return line_data
    
    def compute_all_line_data(self):
        self.all_line_data = []
        for (model, dataset), methods_json_data in self.model_dataset_to_json_data.items():
            for candidate_ttft_slo in self.config.candidate_ttft_slos:
                for candidate_tpot_slo in self.config.candidate_tpot_slos:
                    for method_json_data in methods_json_data:
                        method_name = method_json_data['method_name']
                        line_data = self.compute_line_data(
                            method_json_data, 
                            slo_settings=(candidate_ttft_slo, candidate_tpot_slo), 
                        )
                        self.all_line_data.append(line_data)
                        self.model_dataset_ttft_tpot_method_to_line_data[(model, dataset, candidate_ttft_slo, candidate_tpot_slo, method_name)].append(line_data)

    def criterion(self, line_data: LineData) -> float:
        score1 = sum(line_data.ttft_slo_attainment)
        score2 = sum(line_data.tpot_slo_attainment)
        score3 = sum(line_data.slo_attainment)
        score = score3
        return score
    
    def select_best_line_data(self, candidate_line_datas: list[LineData]) -> LineData:
        best_line_data = max(candidate_line_datas, key=self.criterion)
        return best_line_data
                        
    def compute_best_line_data(self):
        self.model_dataset_ttft_tpot_method_to_best_line_data = {}
        for (model, dataset, candidate_ttft_slo, candidate_tpot_slo, method_name), line_data_with_different_configs in self.model_dataset_ttft_tpot_method_to_line_data.items():
            line_data = self.select_best_line_data(line_data_with_different_configs)
            self.model_dataset_ttft_tpot_method_to_best_line_data[(model, dataset, candidate_ttft_slo, candidate_tpot_slo, method_name)] = line_data
            
    def var_of_lines(self, methods_line_data: list[LineData]):
        data = np.vstack([method_line_data.slo_attainment for method_line_data in methods_line_data]) # (n_methods, n_request_rate)
        var_per_x = np.var(data, axis=0, ddof=0) # (n_request_rate)
        avg_var = np.mean(var_per_x)
        return avg_var
            
    def select_best_slo_settings(self):
        self.compute_all_line_data()
        self.compute_best_line_data()
        self.all_plot_line_data = []
        total_num_request_rates = len(self.all_line_data[0].request_rates)
        for model in self.models:
            for dataset in self.datasets:
                candidate_line_data: dict[tuple[float, float], list[LineData]] = defaultdict(list)
                for TTFT_SLO in self.config.candidate_ttft_slos:
                    for TPOT_SLO in self.config.candidate_tpot_slos:
                        for method_name in self.method_names:
                            line_data = self.model_dataset_ttft_tpot_method_to_best_line_data.get((model, dataset, TTFT_SLO, TPOT_SLO, method_name), None)
                            if line_data:
                                candidate_line_data[(TTFT_SLO, TPOT_SLO)].append(line_data)
                                
                if (model, dataset) in self.config.force_slo_settings:
                    ttft_slo, tpot_slo = self.config.force_slo_settings[(model, dataset)]
                    self.all_plot_line_data.extend(candidate_line_data[ttft_slo, tpot_slo])
                    self.best_slo_settings[(model, dataset)] = ttft_slo, tpot_slo
                else:
                    # select ttft_slo and tpot_slo
                    best_slo_settings: tuple[float, float] = (self.config.candidate_ttft_slos[0], self.config.candidate_tpot_slos[0])
                    max_avg_var = 0.
                    alternative_slo_setteings: list[tuple[float, float]] = []
                    for slo_setting, methods_line_data in candidate_line_data.items():
                        ttft_slo, tpot_slo = slo_setting
                        best_line_data = max(methods_line_data, key=self.criterion)
                        # to better display the difference between methods we want all line go down in the end
                        if best_line_data.goodput == total_num_request_rates:
                            alternative_slo_setteings.append(slo_setting)
                            continue
                        avg_var = self.var_of_lines(methods_line_data)
                        if avg_var > max_avg_var:
                            max_avg_var = avg_var
                            best_slo_settings = (ttft_slo, tpot_slo)
                    self.all_plot_line_data.extend(candidate_line_data[best_slo_settings])
                    self.best_slo_settings[(model, dataset)] = best_slo_settings
                    
    
    def set_dataset_slo_aware_settings(self, dataset_to_slo_settings: dict[str, tuple[float, float]]):
        self.all_line_data = []
        for (model, dataset), methods_json_data in self.model_dataset_to_json_data.items():
            for method_json_data in methods_json_data:
                method_name = method_json_data['method_name']
                line_data = self.compute_line_data(
                    method_json_data, 
                    slo_settings=dataset_to_slo_settings
                )
                self.all_line_data.append(line_data)
        # todo select best config for same method
        self.all_plot_line_data = self.all_line_data
        

    def tabulate_slo(self):
        table_headers = ["Model", "Dataset", "TTFT (s)", "TPOT (s)"]
        table_data = []
        for model in self.models:
            for dataset in self.datasets:
                ttft_slo, tpot_slo = self.best_slo_settings[(model, dataset)]
                model_label = self.config.model_label_map.get(model, model)
                table_data.append((model_label, dataset, ttft_slo, tpot_slo))
        latex_table = tabulate(table_data, headers=table_headers, tablefmt="latex")
        with open(os.path.join(self.tables_dir, "slo_settings.tex"), "w", encoding="utf-8") as f:
            f.write(latex_table)
        print(tabulate(table_data, headers=table_headers, tablefmt='plain'))
        
    def smooth_curve(self, x: list[float], y: list[float], window_size=2):
        y_smooth = []
        for i in range(len(y)):
            start = max(0, i - window_size)
            end = min(i + window_size + 1, len(y))
            window = y[start:end]
            avg = sum(window) / len(window)
            y_smooth.append(avg)
        return x, y_smooth
    
    def draw_slo_vertical_line(self, x_values, y_values, slo_target, ax, **kwargs):
        cross_points = self.get_intersection_points(x_values, y_values, slo_target)
        if not cross_points:
            if y_values[0] < slo_target:
                ax.plot([x_values[0], x_values[0]], [-5, y_values[0]], **kwargs)
            elif y_values[-1] > slo_target:
                ax.plot([x_values[-1], x_values[-1]], [-5, y_values[-1]], **kwargs)
        for i, (x, y) in enumerate(cross_points):
            if i == len(cross_points) - 1:
                ax.plot([x, x], [0, y], **kwargs)
                
    def plot_slo_attainment(self, figure_name: str="slo_attainment"):
        models = list({line_data.model for line_data in self.all_plot_line_data})
        model_id = {model: i for i, model in enumerate(models)}
        id_model = {i: model for i, model in enumerate(models)}
        n_models = len(model_id)
        print(f'model_id: {model_id}')
        
        datasets = list({line_data.dataset for line_data in self.all_plot_line_data})
        datasets = sorted(datasets)
        dataset_id = {dataset: i for i, dataset in enumerate(datasets)}
        id_dataset = {i: dataset for i, dataset in enumerate(datasets)}
        print(f'dataset_id: {dataset_id}')
        n_datasets = len(dataset_id)

        methods = list({line_data.method_name for line_data in self.all_plot_line_data})
        methods = sorted(methods)
        method_id = {method: i for i, method in enumerate(methods)}
        id_method = {i: method for i, method in enumerate(methods)}
        print(f'method_id: {method_id}')
        n_methods = len(methods)
        
        metric_id = {"slo_attainment": 0, "tpot_slo_attainment": 1, "ttft_slo_attainment": 2}
        metric_id_label = {0: "SLO Attainment", 1: "TPOT SLO Attainment",2: "TTFT SLO Attainment"}
        # metric_id = {"slo_attainment": 0}
        # metric_id_label = {0: "SLO Attainment"}
        # metric_id = {"tpot_slo_attainment": 0}
        # metric_id_label = {0: "TPOT SLO Attainment"}
        id_metric = {id: metric for metric, id in metric_id.items()}
        n_metrics = len(metric_id)
        marker_list = ['o', 's', '^', 'v', '>', '<', 'd', 'p', '*', 'h', 'H', 'x', '+', '.', ',', '|', '_']
        color_list = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"]
        n_rows = n_models * n_metrics
        n_cols = n_datasets
        figsize = (6 * n_cols, 4 * n_rows)
        fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
        axes = np.array(axes).reshape(n_rows * n_cols)
        def compute_ax_id(model_id: int, dataset_id: int, metric_id: int) -> int:
            axes_stride = (n_metrics * n_datasets, n_datasets, 1)
            ax_id = model_id * axes_stride[0] + metric_id * axes_stride[1] + dataset_id * axes_stride[2]
            return ax_id

        for line_data in self.all_plot_line_data:
            line_data_model_id = model_id[line_data.model]
            line_data_dataset_id = dataset_id[line_data.dataset]
            line_data_method_id = method_id[line_data.method_name]
            color = color_list[line_data_method_id]
            marker = marker_list[line_data_method_id]

            for k, metric in id_metric.items():
                x = line_data.request_rates
                y = getattr(line_data, metric)
                x, y = self.smooth_curve(x, y)
                ax = axes[compute_ax_id(line_data_model_id, line_data_dataset_id, k)]
                ax.plot(x, y, color=color, marker=marker)
                self.draw_slo_vertical_line(x, y, slo_target=0.9, ax=ax, color=color, linestyle='--', alpha=0.75)

        fontsize=22
        for i in range(n_models):
            for j in range(n_datasets):
                for k in range(n_metrics):
                    ax = axes[compute_ax_id(i, j, k)]
                    ax.grid(False)
                    ax.set_ylim(0, 1.05)

                    formatter = FuncFormatter(lambda val, pos: f'{val * 100:.0f}')
                    ax.yaxis.set_major_formatter(formatter)
                    ax.axhline(y=0.9, color="gray", linestyle="--")

                    metric = id_metric[k]
                    if j == 0:
                        ax.set_ylabel(metric_id_label[k], fontsize=fontsize)
                        ax.text(-0.30, 0.5, id_model[i], transform=ax.transAxes, ha='right', va='center', rotation=90, fontsize=fontsize)
                    if i == n_models - 1 and k == n_metrics - 1:
                        ax.set_xlabel('Request Rate (req/s)', fontsize=fontsize)
                        ax.text(7, -0.45, id_dataset[j], ha='center', va='bottom', fontsize=fontsize)

                    for label in ax.get_xticklabels():
                        label.set_fontsize(fontsize - 5)
                    for label in ax.get_yticklabels():
                        label.set_fontsize(fontsize - 5)
                    ax.tick_params(axis='x', which='major', length=2, width=1, direction='out', grid_color='black', grid_alpha=1)
                    ax.tick_params(axis='y', which='major', length=2, width=1, direction='out', grid_color='black', grid_alpha=1)
                    ax.tick_params(which='both', bottom=True, top=False, left=True, right=False, labelbottom=True, labelleft=True, direction='out')
                    for spine in ax.spines.values():
                        spine.set_edgecolor('black')

        legend_labels = [method for method, id in method_id.items()]
        legend_lines = [Line2D([0], [0], color=color_list[i], marker=marker_list[i]) for i in range(len(method_id))]
        legend_n_cols = len(method_id)

        fig.legend(
            legend_lines,
            legend_labels,
            loc='upper center', ncol=legend_n_cols, fontsize=fontsize + 2, frameon=False, bbox_to_anchor=(0.5, 1.0))

        fig.savefig(os.path.join(self.figures_dir, f"{figure_name}.pdf"), bbox_inches="tight")

In [ ]:
config = ResultAnalyzerConfig(
    model_label_map = {
        "llava-hf/llava-1.5-7b-hf" : "LLaVA-1.5-7B",
        "llava-hf/llava-v1.6-vicuna-7b-hf" : "LLaVA-NeXT-7B",
        "Qwen/Qwen2-VL-7B" : "Qwen2-VL-7B",
        "deepseek-ai/deepseek-vl2-tiny" : "deepseek-vl2-tiny",
    },
    mixed_dataset_label = "Mixed",
    dataset_label_map = {
        "vizwiz_vqa" : "VizWiz",
        "text_vqa" : "TextVQA",
        "mme" : "MME",
        "pope" : "POPE",
        "textcaps" : "TextCaps",
    },
    candidate_ttft_slos = [0.6, 0.8],
    candidate_tpot_slos = [0.04],
)
analyzer = ResultAnalyzer(config)
analyzer.load_result_json_data()
analyzer.classify_filter_model()
analyzer.compute_best_line_data()
analyzer.select_best_slo_settings()
analyzer.log()

In [ ]:
# analyzer.tabulate_slo()

In [ ]:
# analyzer.plot_slo_attainment()

In [ ]:
def format_floats(values: list[float], sig: int = 3, sep: str = " ") -> str:
    return sep.join(f"{v:2.{sig}f}" for v in values)

dataset_to_slo_settings = {
    "lmms-lab/VizWiz-VQA" : (0.9, 0.08),
    "lmms-lab/textvqa" : (0.9, 0.08),
    "lmms-lab/MME" : (0.5, 0.08),
    "lmms-lab/POPE" : (0.9, 0.08),
    "lmms-lab/TextCaps" : (0.9, 0.08),
}

for line_data in analyzer.all_line_data:
    print(f'{line_data.method_name:>10} ttft {format_floats(line_data.avg_ttft)} tpot {format_floats(line_data.p90_tpot)}')

for line_data in analyzer.all_line_data:
    print(f'{line_data.method_name:>10} mme ttft {format_floats(line_data.dataset_to_avg_ttft["lmms-lab/MME"])}')

for line_data in analyzer.all_line_data:
    print(f'{line_data.method_name:>10} textvqa ttft {format_floats(line_data.dataset_to_avg_ttft["lmms-lab/textvqa"])}')

for line_data in analyzer.all_line_data:
    print(f'{line_data.method_name:>10} mme tpot {format_floats(line_data.dataset_to_p90_tpot["lmms-lab/MME"])}')

analyzer.set_dataset_slo_aware_settings(dataset_to_slo_settings=dataset_to_slo_settings)
analyzer.plot_slo_attainment()